# DataMate — chạy bằng vLLM trên Kaggle

Notebook tự dò mã nguồn và dữ liệu trong `/kaggle/input`, chạy vLLM tensor parallel trên **2 GPU**. Thứ tự chạy: cài vLLM, cài TorchAudio khớp CUDA 13.0, nếu Kaggle yêu cầu thì Restart Session, chạy cell kiểm tra import, rồi chạy hai cell pipeline bên dưới. Mặc định chạy **3 câu hỏi** để thử; đổi `LIMIT = None` để chạy toàn bộ. Bật GPU và Internet.

In [ ]:
%pip install -q vllm==0.32.0

In [ ]:
%pip install --force-reinstall --no-deps torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130

In [ ]:
import torch
import torchaudio
from vllm import LLM, SamplingParams

print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('TorchAudio:', torchaudio.__version__)
print('vLLM import OK')
print('GPU count:', torch.cuda.device_count())
assert torch.version.cuda == '13.0', 'Cell cài TorchAudio hiện đặt cho CUDA 13.0; kiểm tra lại runtime CUDA.'
assert torch.cuda.device_count() >= 2, 'Notebook cần được cấp ít nhất 2 GPU.'

In [ ]:
from pathlib import Path
import os
import sys

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/result')
SPLIT = 'public_test'  # đổi thành 'train' nếu muốn chạy trên tập train
LIMIT = 3              # đặt None để chạy tất cả câu hỏi
MODEL_DIR = 'Qwen/Qwen2.5-Coder-3B-Instruct'
TENSOR_PARALLEL_SIZE = 2

# Nếu tự dò không đúng, điền đường dẫn tương ứng từ mục Input ở đây.
CODE_DIR = None
DATA_DIR = None

assert INPUT_ROOT.is_dir(), f'Không tìm thấy Kaggle Input: {INPUT_ROOT}'

def find_project_and_data(input_root):
    code_candidates = []
    data_candidates = []
    for root, dirs, files in os.walk(input_root):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        current = Path(root)
        if (current / 'src' / 'predict.py').is_file() and (current / 'prompts').is_dir():
            code_candidates.append(current)
        if ('questions.jsonl' in files and 'calendar.csv' in files
                and 'rulebook.md' in files and (current / 'data').is_dir()):
            data_candidates.append(current)

    if CODE_DIR is None:
        if not code_candidates:
            raise FileNotFoundError('Không tìm thấy thư mục có src/predict.py và prompts/ trong /kaggle/input. Hãy đặt CODE_DIR thủ công.')
        code_path = code_candidates[0]
    else:
        code_path = Path(CODE_DIR)

    if DATA_DIR is None:
        preferred = [p for p in data_candidates if SPLIT in p.parts]
        if not preferred:
            preferred = [p for p in data_candidates if p.name == SPLIT]
        if not preferred:
            raise FileNotFoundError(f'Không tìm thấy thư mục dữ liệu {SPLIT!r} có questions.jsonl, calendar.csv, rulebook.md và data/. Hãy đặt DATA_DIR thủ công.')
        data_path = preferred[0]
    else:
        data_path = Path(DATA_DIR)

    return code_path.resolve(), data_path.resolve()

code_path, data_path = find_project_and_data(INPUT_ROOT)
assert (code_path / 'src' / 'predict.py').is_file(), f'CODE_DIR không hợp lệ: {code_path}'
assert (data_path / 'questions.jsonl').is_file(), f'DATA_DIR không hợp lệ: {data_path}'
print('Code:', code_path)
print('Data:', data_path)
print('Output:', OUTPUT_DIR)

In [ ]:
sys.path.insert(0, str(code_path))
from src.predict import DataMatePipeline
from src.modules.answer_formatter import build_submission_line
import src.predict as predict_module
import json
import logging
import math
from datetime import date, datetime
import numpy as np
import pandas as pd

def json_safe(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, pd.Period):
        return str(value)
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (pd.Series, pd.Index)):
        return json_safe(value.tolist())
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f'Không thể chuyển {type(value).__name__} sang JSON')

# Keep the unwrapped function on the module so rerunning this cell stays safe.
if not hasattr(predict_module, '_json_safe_original_format_answer'):
    predict_module._json_safe_original_format_answer = predict_module.format_answer
_original_format_answer = predict_module._json_safe_original_format_answer

def _json_safe_format_answer(*args, **kwargs):
    return json_safe(_original_format_answer(*args, **kwargs))

predict_module.format_answer = _json_safe_format_answer
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

with (data_path / 'questions.jsonl').open(encoding='utf-8') as f:
    questions = [json.loads(line) for line in f if line.strip()]
questions.sort(key=lambda q: q['business_id'])
if LIMIT is not None:
    questions = questions[:LIMIT]
if not questions:
    raise ValueError(f'Không có câu hỏi trong {data_path / "questions.jsonl"}')

import torch
if torch.cuda.device_count() < TENSOR_PARALLEL_SIZE:
    raise RuntimeError(
        f'Cần {TENSOR_PARALLEL_SIZE} GPU để chạy tensor parallel, '
        f'nhưng Kaggle chỉ cấp {torch.cuda.device_count()} GPU.'
    )

# Khởi tạo vLLM trực tiếp để lỗi khởi tạo hiện rõ, không âm thầm fallback sang Transformers.
from vllm import LLM, SamplingParams
print(f'Khởi tạo vLLM trên {TENSOR_PARALLEL_SIZE} GPU: {MODEL_DIR}')
llm = LLM(
    model=MODEL_DIR,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    dtype='float16',
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
    stop=['```\n', '\n\n\n'],
    include_stop_str_in_output=True,
)

def generate_with_vllm(prompt):
    if getattr(tokenizer, 'chat_template', None):
        prompt = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    outputs = llm.generate([prompt], sampling_params)
    return outputs[0].outputs[0].text

print(f'vLLM sẵn sàng; sẽ chạy {len(questions)} câu hỏi trên '
      f'{torch.cuda.device_count()} GPU.')
pipeline = DataMatePipeline(
    data_dir=str(data_path),
    output_dir=str(OUTPUT_DIR),
    model_dir=MODEL_DIR,
    max_retries=2,
    confidence_threshold=0.35,
    llm_generate_fn=generate_with_vllm,
)

submission_path = OUTPUT_DIR / 'submission.jsonl'
if submission_path.exists():
    submission_path.unlink()

ANSWER_TYPES = {'number', 'percent', 'string', 'date', 'boolean', 'list', 'table'}
SUBMISSION_TYPES = ANSWER_TYPES | {'unanswerable', 'abstain'}

def make_submission_line(question_id, answer):
    if not isinstance(answer, dict) or answer.get('type') not in SUBMISSION_TYPES:
        logging.warning('Invalid answer for %s; writing abstain.', question_id)
        answer = {'type': 'abstain'}

    answer_type = answer['type']
    record = {'id': str(question_id), 'type': answer_type}
    if answer_type in ANSWER_TYPES:
        value = json_safe(answer.get('answer'))
        valid = value is not None
        if answer_type in {'number', 'percent'}:
            valid = valid and isinstance(value, (int, float)) and not isinstance(value, bool)
        elif answer_type in {'string', 'date'}:
            valid = valid and isinstance(value, str)
        elif answer_type == 'boolean':
            valid = valid and isinstance(value, bool)
        elif answer_type == 'list':
            valid = valid and isinstance(value, list)
        elif answer_type == 'table':
            valid = (
                valid and isinstance(value, dict)
                and isinstance(value.get('columns'), list)
                and isinstance(value.get('rows'), list)
                and all(isinstance(row, list) for row in value.get('rows', []))
            )

        if valid:
            record['answer'] = value
        else:
            logging.warning('Invalid %s answer for %s; writing abstain.', answer_type, question_id)
            record = {'id': str(question_id), 'type': 'abstain'}
    elif isinstance(answer.get('reason'), str) and answer['reason'].strip():
        record['reason'] = answer['reason'].strip()

    return build_submission_line(record.pop('id'), record)

with submission_path.open('w', encoding='utf-8') as out:
    for index, question in enumerate(questions, start=1):
        print(f'[{index}/{len(questions)}] {question["id"]}: {question["question"]}')
        try:
            answer = pipeline.process_question(question)
            line = make_submission_line(question['id'], answer)
        except Exception:
            logging.exception('Question %s failed; writing abstain and continuing.', question['id'])
            line = build_submission_line(question['id'], {'type': 'abstain'})
        out.write(line + '\n')
        out.flush()

records = [
    json.loads(line)
    for line in submission_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
expected_ids = [str(question['id']) for question in questions]
actual_ids = [record.get('id') for record in records]
if actual_ids != expected_ids:
    raise ValueError('Submission JSONL thiếu, thừa hoặc sai thứ tự ID câu hỏi.')
for record in records:
    answer_type = record.get('type')
    allowed_keys = {'id', 'type', 'reason'}
    required_keys = {'id', 'type'}
    if answer_type in ANSWER_TYPES:
        allowed_keys.add('answer')
        required_keys.add('answer')
    if (answer_type not in SUBMISSION_TYPES
            or not required_keys.issubset(record)
            or not set(record).issubset(allowed_keys)):
        raise ValueError(f'Bản ghi submission sai format: {record!r}')

print(f'Hoàn tất. Submission: {submission_path}')
print(submission_path.read_text(encoding='utf-8'))